# 00 — Download Video Data (serverless)

Downloads the public Kaggle CCTV anomaly dataset, stages it to a Unity Catalog Volume, and builds a Delta reference table with per-file duration metadata.

**Compute:** serverless CPU (environment version 4 or later). No classic cluster, no init scripts, no `apt-get`.

Differences from the classic version (`3_Video_Batch_Inference_Ray_Core`):
* `ffprobe` (system binary installed with `apt-get`) is replaced by [PyAV](https://pyav.basswood-io.com/), a pip package that bundles FFmpeg, so it works on serverless executors.
* Kaggle files are staged on local disk and then moved into the Volume.

**Prerequisite — Kaggle API token.** Either:
1. **Databricks secrets (recommended).** Secret scope `kaggle` with keys `username` and `key` (set the `KAGGLE_SECRET_SCOPE` widget if your scope has a different name).
2. **`~/.kaggle/kaggle.json`** in the notebook user's home directory.

In [0]:
%pip install -q av==14.0.1 kaggle
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("CATALOG", "main", label="CATALOG")
# Use the same CATALOG/SCHEMA/VOLUME in notebooks 00-04. This notebook creates them if missing.
dbutils.widgets.text("SCHEMA", "vlm_video", label="SCHEMA")
dbutils.widgets.text("VOLUME", "transcribe-video", label="VOLUME")
dbutils.widgets.text("CATEGORY_FILTER", "shoplifting", label="CATEGORY_FILTER")
dbutils.widgets.text("KAGGLE_SECRET_SCOPE", "kaggle", label="KAGGLE_SECRET_SCOPE")

CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
VOLUME = dbutils.widgets.get("VOLUME")
CATEGORY_FILTER = dbutils.widgets.get("CATEGORY_FILTER")
KAGGLE_SECRET_SCOPE = dbutils.widgets.get("KAGGLE_SECRET_SCOPE")
print(f"Target: {CATALOG}.{SCHEMA}.{VOLUME} · category: {CATEGORY_FILTER}")

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`")

import os
video_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/raw_video/CCTV"
os.makedirs(video_path, exist_ok=True)
print(video_path)

In [0]:
import os, sys, shutil
from pathlib import Path

try:
    os.environ["KAGGLE_USERNAME"] = dbutils.secrets.get(scope=KAGGLE_SECRET_SCOPE, key="username")
    os.environ["KAGGLE_KEY"] = dbutils.secrets.get(scope=KAGGLE_SECRET_SCOPE, key="key")
    print(f"Kaggle credentials loaded from secret scope '{KAGGLE_SECRET_SCOPE}'.")
except Exception:
    if not (Path.home() / ".kaggle" / "kaggle.json").exists():
        raise RuntimeError(
            f"No Kaggle credentials found. Create secret scope '{KAGGLE_SECRET_SCOPE}' with keys "
            "'username' and 'key', or place kaggle.json in ~/.kaggle/. See https://www.kaggle.com/docs/api"
        )
    print("Kaggle credentials loaded from ~/.kaggle/kaggle.json.")

# On serverless the pip-installed CLI lives next to the notebook's Python, not on PATH.
KAGGLE_BIN = shutil.which("kaggle") or os.path.join(os.path.dirname(sys.executable), "kaggle")
assert os.path.exists(KAGGLE_BIN), f"kaggle CLI not found at {KAGGLE_BIN}"
print(KAGGLE_BIN)

### Download the dataset
Public Kaggle dataset: [Real-Time Anomaly Detection in CCTV Surveillance](https://www.kaggle.com/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data).

Set `CATEGORY_FILTER` to one anomaly category (e.g. `shoplifting`, `burglary`, `arson`) to download only that folder, file by file. Set it to `ALL` to download the full ~95 GB dataset. One category takes ~10 min and ~2 GB.

In [0]:
import subprocess, tempfile, zipfile

DATASET = "webadvisor/real-time-anomaly-detection-in-cctv-surveillance"

if CATEGORY_FILTER.upper() == "ALL":
    # Full dataset (~95 GB) goes straight to the Volume; local disk is too small.
    result = subprocess.run(
        [KAGGLE_BIN, "datasets", "download", "-d", DATASET, "-p", video_path, "--unzip"],
        capture_output=True, text=True,
    )
    print(result.stdout[-2000:])
    if result.returncode != 0:
        raise RuntimeError(f"Kaggle download failed: {result.stderr}")
else:
    # The Kaggle list_files API only returns the first page for large datasets, so
    # enumerate by the dataset's <Category>NNN_x264.mp4 naming pattern instead.
    # Missing numbers return 404 and are skipped.
    staging = tempfile.mkdtemp(prefix="kaggle_")
    capitalised = CATEGORY_FILTER[0].upper() + CATEGORY_FILTER[1:]
    candidate_paths = [f"data/{CATEGORY_FILTER}/{capitalised}{i:03d}_x264.mp4" for i in range(1, 200)]
    print(f"Attempting up to {len(candidate_paths)} files from data/{CATEGORY_FILTER}/ (staging in {staging})")

    def staged_files():
        # Kaggle CLI versions differ: files may land flat, nested under data/<category>/,
        # or zipped. Walk the whole staging tree.
        return {os.path.join(dp, f) for dp, _, fs in os.walk(staging) for f in fs}

    success_count = fail_count = 0
    for i, fname in enumerate(candidate_paths, 1):
        before = staged_files()
        proc = subprocess.run(
            [KAGGLE_BIN, "datasets", "download", "-d", DATASET, "-f", fname, "-p", staging, "--force"],
            capture_output=True, text=True,
        )
        new_files = staged_files() - before
        # Count a success only if a file actually appeared; some CLI versions exit 0 on 404.
        if new_files:
            success_count += 1
            if success_count == 1:
                print(f"  First download landed as: {sorted(new_files)}")
            elif success_count % 10 == 0:
                print(f"  [{success_count} downloaded] last: {fname}")
        else:
            fail_count += 1
            msg = f"{proc.stdout} {proc.stderr}".lower()
            if i == 1 or ("404" not in msg and "not found" not in msg):
                print(f"  WARN: {fname}: rc={proc.returncode} {(proc.stdout + proc.stderr).strip()[:300]}")
        # Stop after 30 consecutive misses once at least one file has been found.
        if success_count > 0 and i - success_count > 30:
            print(f"  Stopping early after {i} attempts ({success_count} downloaded)")
            break

    if success_count == 0:
        raise RuntimeError(
            f"No files downloaded for category '{CATEGORY_FILTER}'. Check the spelling "
            "(shoplifting, burglary, abuse, arson, assault, robbery, stealing, vandalism) "
            "or set CATEGORY_FILTER=ALL."
        )

    # Single-file downloads arrive flat, sometimes as .mp4.zip. Unzip, then move into
    # data/<category>/ so the layout matches what notebook 02 expects.
    for zpath in [p for p in staged_files() if p.endswith(".zip")]:
        try:
            with zipfile.ZipFile(zpath) as z:
                z.extractall(os.path.dirname(zpath))
            os.remove(zpath)
        except zipfile.BadZipFile:
            print(f"  WARN: bad zip {zpath}")

    target_dir = f"{video_path}/data/{CATEGORY_FILTER}"
    os.makedirs(target_dir, exist_ok=True)
    mp4s = sorted(p for p in staged_files() if p.lower().endswith(".mp4"))
    for p in mp4s:
        shutil.copyfile(p, os.path.join(target_dir, os.path.basename(p)))
    shutil.rmtree(staging, ignore_errors=True)
    print(f"Downloaded {success_count} files; staged {len(mp4s)} .mp4 files into {target_dir}")
    if not mp4s:
        raise RuntimeError("Downloads reported success but no .mp4 files were found; see the WARN lines above.")

In [0]:
import pyspark.sql.functions as F
from functools import reduce

# One listing per category folder, unioned into a single DataFrame.
category_dirs = [i.path for i in dbutils.fs.ls(f"{video_path}/data/")]

def list_category(folder_path):
    return (
        spark.createDataFrame(dbutils.fs.ls(folder_path))
        .withColumn("file_path", F.expr("substring(path, 6, length(path))"))  # strip 'dbfs:'
    )

# Skip empty category folders (createDataFrame can't infer a schema from an empty list).
non_empty = [d for d in category_dirs if dbutils.fs.ls(d)]
if not non_empty:
    raise RuntimeError(f"No video files under {video_path}/data/ — re-run the download cell.")
file_reference_df = reduce(lambda a, b: a.union(b), map(list_category, non_empty))
file_reference_df = file_reference_df.filter(F.col("file_path").endswith(".mp4"))
print(f"{file_reference_df.count()} .mp4 files found")

In [0]:
from pyspark.sql.types import FloatType

@F.udf(FloatType())
def get_length_udf(filename):
    """Container duration in seconds via PyAV (bundled FFmpeg). None if unreadable."""
    import av
    try:
        with av.open(filename) as c:
            if c.duration:
                return float(c.duration / 1e6)
            vs = c.streams.video[0]
            return float(vs.duration * vs.time_base) if vs.duration else None
    except Exception:
        return None

file_reference_df = file_reference_df.withColumn("duration_seconds", get_length_udf("file_path"))

In [0]:
(file_reference_df.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.videos_file_reference"))

ref = spark.table(f"{CATALOG}.{SCHEMA}.videos_file_reference")
nulls = ref.filter(F.col("duration_seconds").isNull()).count()
print(f"{ref.count()} rows written · {nulls} with null duration")
display(ref)

## Exploratory Data Analysis

In [0]:
display(ref.selectExpr(
    "percentile(duration_seconds, 0.0) as min",
    "percentile(duration_seconds, 0.25) as Q1",
    "percentile(duration_seconds, 0.5) as median",
    "percentile(duration_seconds, 0.75) as Q3",
    "percentile(duration_seconds, 1.0) as max",
))

In [0]:
import matplotlib.pyplot as plt

durations = ref.select("duration_seconds").toPandas()["duration_seconds"].dropna()
plt.figure(figsize=(10, 5))
plt.hist(durations, bins=30)
plt.title("Video durations")
plt.xlabel("Duration (seconds)")
plt.ylabel("Count")
plt.show()